In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using StaticArrays
using FastChebInterp
using BenchmarkTools
using JLD2
using Test

include("Chebyshev.jl")
using .Chebyshev
import .Chebyshev: normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw

In [3]:
function fᵤ(u)
    x = u^2
    return sin(x)
end

function f(x::SVector{1, Float64})
    return sin.(x)
end

function ∇f(x::SVector{1, Float64})
    return cos.(x)
end

function Hf(x::SVector{1, Float64})
    return -sin.(x)
end

function u(x::SVector{1, Float64})
    return x.^0.5
end

function ∇u(x::SVector{1, Float64})
    return reshape(0.5*x.^-0.5, Size(1, 1))
end

function Hu(x::SVector{1, Float64})
    return reshape(-0.25*x.^-1.5, Size(1, 1, 1))
end

function u_to_x(u::SVector{1, Float64})
    return u.^2
end

x1, x2 = SA[0.0], SA[0.5*π]
u1, u2 = u(x1), u(x2)

n = 50

uc = chebpoints(n, u1[], u2[])
cb = chebinterp(fᵤ.(uc), u1[], u2[])
cs = ChebyshevSeries(cb.coefs, u1[], u2[])
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu);

# coefs = cb.coefs
# @save "test_chebyshev_1dts.jld2" coefs;

In [4]:
x3 = x1 + rand()*(x2 - x1)
u3 = u(x3)
contains(cs, u3) || error("u3 not in cs");

In [5]:
println(ts(x3[]) - f(x3)[])

-2.0816681711721685e-17


In [6]:
g, ∇g = gradient(ts, x3[])

println(g - f(x3)[])
println(∇g - ∇f(x3)[])

-2.0816681711721685e-17
-4.6629367034256575e-15


In [7]:
g, ∇g, Hg = hessian(ts, x3[])

println(g - f(x3)[])
println(∇g - ∇f(x3)[])
println(Hg - Hf(x3)[])

-2.0816681711721685e-17
-4.6629367034256575e-15
-5.2534365746481626e-14


In [3]:
function fᵤ(u::SVector{2, Float64})
    r, θ = u
    return exp(r*cos(θ)) * cos(r*sin(θ))
end

function f(x::SVector{2, Float64})
    ξ, η = x
    return exp(ξ) * cos(η)
end

function ∇f(x::SVector{2, Float64})
    ξ, η = x

    f₁ = exp(ξ)*cos(η)
    f₂ = -exp(ξ)*sin(η)
    
    return SA[f₁, f₂]
end

function Hf(x::SVector{2, Float64})
    ξ, η = x

    f₁₁ = exp(ξ)*cos(η)
    f₁₂ = -exp(ξ)*sin(η)
    f₂₁ = f₁₂
    f₂₂ = -exp(ξ)*cos(η)
    
    return SA[f₁₁ f₁₂;
              f₂₁ f₂₂]
end

function u(x::SVector{2, Float64})
    ξ, η = x

    r = sqrt(ξ^2 + η^2)
    θ = atan(η/ξ)
    
    return SA[r, θ]
end

function ∇u(x::SVector{2, Float64})
    ξ, η = x
    
    r² = ξ^2 + η^2
    r = √r²

    r₁ = ξ/r
    r₂ = η/r
    θ₁ = -η/r²
    θ₂ = ξ/r²

    ∇u₁ = SA[r₁, r₂]
    ∇u₂ = SA[θ₁, θ₂]
    
    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2, Float64})
    ξ, η = x

    ξ², η² = ξ^2, η^2
    
    r² = ξ² + η²
    r = √r²
    r³ = r² * r
    r⁴ = r³ * r

    r₁₁ = η²/r³
    r₁₂ = -ξ*η/r³
    r₂₁ = r₁₂
    r₂₂ = ξ²/r³
    
    θ₁₁ = 2*ξ*η/r⁴
    θ₁₂ = (η² - ξ²)/r⁴
    θ₂₁ = θ₁₂
    θ₂₂ = -2*ξ*η/r⁴

    Hu₁ = reshape([r₁₁ r₁₂;
                   r₂₁ r₂₂], 1, 2, 2)
    
    Hu₂ = reshape([θ₁₁ θ₁₂;
                   θ₂₁ θ₂₂], 1, 2, 2)

    hess_u = vcat(Hu₁, Hu₂)
    
    return SArray{Tuple{2, 2, 2}, Float64}(hess_u)
end

function u_to_x(u::SVector{2, Float64})
    r, θ = u
    return SA[r*cos(θ), r*sin(θ)]
end

u1 = SA[0.1, -0.2*π]
u2 = SA[2.0, 0.4*π]

n = (50, 50)

uc = chebpoints(n, u1, u2)
cb = chebinterp(fᵤ.(uc), u1, u2)
cs = ChebyshevSeries(cb.coefs, u1, u2)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu);

# coefs = cb.coefs
# @save "test_chebyshev_2dts.jld2" coefs;

In [4]:
u3 = u1 + rand(SVector{2, Float64}) .* (u2 - u1)
x3 = u_to_x(u3)
contains(cs, u3) || error("u3 not in cs");

In [5]:
x3

2-element SVector{2, Float64} with indices SOneTo(2):
 0.08699625453821141
 0.10996289937250477

In [10]:
g = ts(x3)

ts(x3) - f(x3)

4.440892098500626e-16

In [11]:
g, ∇g = gradient(ts, x3)

println(g - f(x3))
println(∇g - ∇f(x3))

4.440892098500626e-16
[-1.7763568394002505e-14, -2.5757174171303632e-14]


In [12]:
g, ∇g, Hg = hessian(ts, x3)

println(g - f(x3))
println(∇g - ∇f(x3))
println(Hg - Hf(x3))

4.440892098500626e-16
[-1.7763568394002505e-14, -2.5757174171303632e-14]
[1.3811174426336947e-13 6.865619184281968e-13; 6.865619184281968e-13 1.5050183321818622e-12]


In [6]:
function fᵤ(u::SVector{3, Float64})
    r, θ, ϕ = u
    r² = r^2
    return r² * sin(ϕ) * cos(θ) * cos(ϕ) * exp(-r²)
end

function f(x::SVector{3, Float64})
    ξ, η, ζ = x
    return ξ * ζ * exp(-ξ^2 - η^2 - ζ^2)
end

function ∇f(x::SVector{3, Float64})
    ξ, η, ζ = x

    ξ², η², ζ² = ξ^2, η^2, ζ^2
    r² = ξ² + η² + ζ²
    er = exp(-r²)
    
    f₁ = ζ * (1 - 2ξ²) * er
    f₂ = -2*ξ*η*ζ * er
    f₃ = ξ * (1 - 2ζ²) * er
    
    return SA[f₁, f₂, f₃]
end

function Hf(x::SVector{3, Float64})
    ξ, η, ζ = x

    ξ², η², ζ² = ξ^2, η^2, ζ^2
    r² = ξ² + η² + ζ²
    er = exp(-r²)
    
    f₁₁ = 2*ξ*ζ * (2ξ² - 3) * er
    f₁₂ = 2*η*ζ * (2ξ² - 1) * er
    f₁₃ = (4*ξ²*ζ² - 2ξ² - 2ζ² + 1) * er
    f₂₁ = f₁₂
    f₂₂ = 2*ξ*ζ * (2η² - 1) * er
    f₂₃ = 2*ξ*η * (2ζ² - 1) * er
    f₃₁ = f₁₃
    f₃₂ = f₂₃
    f₃₃ = 2*ξ*ζ * (2ζ² - 3) * er
    
    return SA[f₁₁ f₁₂ f₁₃;
              f₂₁ f₂₂ f₂₃;
              f₃₁ f₃₂ f₃₃]
end

function u(x::SVector{3, Float64})
    ξ, η, ζ = x

    ρ = sqrt(ξ^2 + η^2)
    
    r = sqrt(ξ^2 + η^2 + ζ^2)
    θ = atan(η/ξ)
    ϕ = atan(ρ/ζ)
    
    return SA[r, θ, ϕ]
end

function ∇u(x::SVector{3, Float64})
    ξ, η, ζ = x
    
    r² = ξ^2 + η^2 + ζ^2
    r = √r²
    ρ² = ξ^2 + η^2
    ρ = √ρ²
    
    r₁ = ξ/r
    r₂ = η/r
    r₃ = ζ/r
    θ₁ = -η/ρ²
    θ₂ = ξ/ρ²
    θ₃ = 0
    ϕ₁ = ξ*ζ/(ρ*r²)
    ϕ₂ = η*ζ/(ρ*r²)
    ϕ₃ = -ρ/r²

    ∇u₁ = SA[r₁, r₂, r₃]
    ∇u₂ = SA[θ₁, θ₂, θ₃]
    ∇u₃ = SA[ϕ₁, ϕ₂, ϕ₃]
    
    return vcat(∇u₁', ∇u₂', ∇u₃')
end

function Hu(x::SVector{3, Float64})
    ξ, η, ζ = x

    ξ² = ξ^2
    η² = η^2
    ζ² = ζ^2
    
    r² = ξ² + η² + ζ²
    r = √r²
    r³ = r²*r
    r⁴ = r³*r
    
    ρ² = ξ² + η²
    ρ = √ρ²
    ρ³ = ρ²*ρ
    ρ⁴ = ρ³*ρ
    
    r₁₁ = (η² + ζ²) / r³
    r₁₂ = -ξ * η / r³
    r₁₃ = -ξ * ζ / r³
    r₂₁ = r₁₂
    r₂₂ = (ξ² + ζ²) / r³
    r₂₃ = -η * ζ / r³
    r₃₁ = r₁₃
    r₃₂ = r₂₃
    r₃₃ = ρ² / r³

    θ₁₁ = 2 * ξ * η / ρ⁴
    θ₁₂ = (η² - ξ²) / ρ⁴
    θ₁₃ = 0
    θ₂₁ = θ₁₂
    θ₂₂ = -θ₁₁
    θ₂₃ = 0
    θ₃₁ = θ₁₃
    θ₃₂ = θ₂₃
    θ₃₃ = 0
    
    ϕ₁₁ = ζ * (-ξ²*ζ² - 3*ξ²*ρ² + ρ²*r²) / (ρ³ * r⁴)
    ϕ₁₂ = -ξ*η*ζ * (3*ξ² + 3*η² + ζ²) / (ρ³ * r⁴)
    ϕ₁₃ = ξ * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₂₁ = ϕ₁₂
    ϕ₂₂ = ζ * (-η²*ζ² - 3*η²*ρ² + ρ²*r²) / (ρ³ * r⁴)
    ϕ₂₃ = η * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₃₁ = ϕ₁₃
    ϕ₃₂ = ϕ₂₃
    ϕ₃₃ = 2ζ*ρ/r⁴
    
    Hu₁ = reshape([r₁₁ r₁₂ r₁₃;
                   r₂₁ r₂₂ r₂₃;
                   r₃₁ r₃₂ r₃₃], 1, 3, 3)
    
    Hu₂ = reshape([θ₁₁ θ₁₂ θ₁₃;
                   θ₂₁ θ₂₂ θ₂₃;
                   θ₃₁ θ₃₂ θ₃₃], 1, 3, 3)
    
    Hu₃ = reshape([ϕ₁₁ ϕ₁₂ ϕ₁₃;
                   ϕ₂₁ ϕ₂₂ ϕ₂₃;
                   ϕ₃₁ ϕ₃₂ ϕ₃₃], 1, 3, 3)

    hess_u = vcat(Hu₁, Hu₂, Hu₃)
    
    return SArray{Tuple{3, 3, 3}, Float64}(hess_u)
end

function u_to_x(u::SVector{3, Float64})
    r, θ, ϕ = u
    
    ξ = r * sin(ϕ) * cos(θ)
    η = r * sin(ϕ) * sin(θ)
    ζ = r * cos(ϕ)
    
    return SA[ξ, η, ζ]
end

u1 = SA[0.1, 0.2, 0.3]
u2 = SA[2.2, 1.8, 1.6]

n = (50, 50, 50)

uc = chebpoints(n, u1, u2)
cb = chebinterp(fᵤ.(uc), u1, u2)
cs = ChebyshevSeries(cb.coefs, u1, u2)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu);

# coefs = cb.coefs
# @save "test_chebyshev_3dts.jld2" coefs;

In [7]:
u3 = u1 + rand(SVector{3, Float64}) .* (u2 - u1)
x3 = u_to_x(u3)
contains(cs, u3) || error("u3 not in cs");

In [8]:
x3

3-element SVector{3, Float64} with indices SOneTo(3):
 0.39124354777134873
 0.4717891041479673
 0.6073568833909956

In [15]:
g = ts(x3)

g - f(x3)

0.050134395291784345

In [16]:
g, ∇g = gradient(ts, x3)

println(g - f(x3))
println(∇g - ∇f(x3))

0.050134395291784345
[-0.6689767577833274, -0.048090149224169515, 0.006442643985018301]


In [17]:
g, ∇g, Hg = hessian(ts, x3)

println(g - f(x3))
println(∇g - ∇f(x3))
println(Hg - Hf(x3))

0.050134395291784345
[-0.6689767577833274, -0.048090149224169515, 0.006442643985018301]
[-0.2996737904973994 0.641708148635098 -0.08596534123631994; 0.641708148635098 -0.05414223573992609 -0.0061773038500382545; -0.08596534123631991 -0.006177303850038256 -0.209246570249155]
